In [121]:
import math
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

from PIL import Image
from matplotlib.lines import Line2D
from matplotlib.patches import Wedge
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from functools import reduce

In [ ]:
#Specify where you want to import the files from!
import os
print(os.getcwd())
os.chdir(r'C:\Users\USER\Downloads\HTN RESPONSE\htnglobalanalyses')

C:\Users\USER\Downloads\HTN RESPONSE\htnglobalanalyses


In [123]:
#import dataset for men
dfm = pd.read_csv("./data/NCD-RisC_Data.csv", index_col=None)
dfm = dfm.sort_values("Year").reset_index(drop=True)
latest_year = dfm['Year'].max()
df = dfm[dfm['Year'] == latest_year].reset_index(drop=True)
# df2.to_csv("NCDRISK2019_data.csv", index=False)
df.head()
df.to_excel("latest_year_data.xlsx", index=False)

In [124]:
# Split into Men and Women
df2 = pd.read_excel("latest_year_data.xlsx", index_col=None)
df_men = df2[df2['Sex'] == 'Men'].copy()
df_women = df2[df2['Sex'] == 'Women'].copy()

# Sort by 'Country code'
df_men = df_men.sort_values(by='Code').reset_index(drop=True)
df_women = df_women.sort_values(by='Code').reset_index(drop=True)
df_men.columns

Index(['Country', 'Code', 'Sex', 'Year', 'Age', 'htn', 'htn_lcl', 'htn_ucl',
       'treat', 'treat_lcl', 'treat_ucl', 'control', 'control_lcl',
       'control_ucl'],
      dtype='object')

In [125]:
# Select only relevant columns from each DataFrame
df_men_avg = df_men[['Country', 'Code', 'htn', 'htn_lcl', 'htn_ucl']].copy()
df_women_avg = df_women[['Country', 'Code', 'htn', 'htn_lcl', 'htn_ucl']].copy()

# Merge on 'Country code'
df_merged = pd.merge(df_men_avg, df_women_avg, on='Code', suffixes=('_Men', '_Women'))
df_merged.columns

df_merged = df_merged.rename(columns={
    'Country_Men': 'Country',
    'Country_Women': 'Country2'
})


# Compute average values
df_merged['htn_avg'] = df_merged[['htn_Men', 'htn_Women']].mean(axis=1)
df_merged['htn_lcl_avg'] = df_merged[['htn_lcl_Men', 'htn_lcl_Women']].mean(axis=1)
df_merged['htn_ucl_avg'] = df_merged[['htn_ucl_Men', 'htn_ucl_Women']].mean(axis=1)

# Optional: Keep only Country code and the averaged columns
df_avg = df_merged[['Country','Country2', 'Code', 'htn_avg', 'htn_lcl_avg', 'htn_ucl_avg']].copy()
df_avg.to_excel("htn_avg_data.xlsx", index=False)

In [ ]:
#import pop and health workers related data from data_for_map and merge dataframe together
df_map = pd.read_csv("data_for_map.csv", index_col=None)
df_map=df_map.rename(columns={
    'Country Code': 'Code'
})
df_map = df_map.sort_values(by='Code').reset_index(drop=True)
df_map.columns

Index(['Country_x', 'Code', 'Sex_x', 'Year_x', 'htn', 'treat', 'control',
       'diagn', 'untreatedstage2HTN', 'Country_y', 'Income', 'pop', 'pop_tot',
       'Indicator_b', 'Country_b', 'Year_b', 'GeoRegion_b', 'nurse',
       'Indicator_c', 'Country_c', 'Year_c', 'GeoRegion_c', 'pharma',
       'Indicator_d', 'Country_d', 'Year_d', 'GeoRegion_d', 'chws_n',
       'Indicator_a', 'Country_a', 'Year_a', 'phy', 'chws', 'hw', 'treatn',
       'controln', 'diagng', 'htn2', 'n_phy', 'b_phy', 'pop_htn',
       'pop_htn_visit1', 'pop_htn_visit2', 'pop_htn_visit3', 'phy_s1',
       'phy_s2', 'phy_s1_i1', 'phy_s2_i2', 'phy_s1_c', 'phy_s2_c',
       'gaps_hf_base_s1', 'gaps_hf_base_s2', 'gaps_hf_base_s1_i1',
       'gaps_hf_base_s2_i2', 'gaps_hf_base_s1_c', 'gaps_hf_base_s2_c',
       'gaps_hf_6_s1', 'gaps_hf_6_s2', 'gaps_hf_3_s1', 'gaps_hf_3_s2',
       'gaps_hf_6_s1_i1', 'gaps_hf_6_s2_i2', 'gaps_hf_3_s1_i1',
       'gaps_hf_3_s2_i2', 'gaps_hf_6_s1_c', 'gaps_hf_6_s2_c', 'gaps_hf_3_s1_c',
     

In [159]:
# Merge on 'Country code'
df_merged = pd.merge(df_map, df_avg, on='Code',how='left')
df_merged.columns
df_merged[['htn_avg', 'htn_lcl_avg', 'htn_ucl_avg']] = df_merged[['htn_avg', 'htn_lcl_avg', 'htn_ucl_avg']] * 100
len(df_merged)

199

In [128]:
df_merged['pop_htn_lcl']=df_merged['pop']*df_merged['htn_lcl_avg']/100
df_merged['pop_htn_ucl']=df_merged['pop']*df_merged['htn_ucl_avg']/100

In [157]:
df=df_merged #renamed to df
df.columns

Index(['Country_x', 'Code', 'Sex_x', 'Year_x', 'htn', 'treat', 'control',
       'diagn', 'untreatedstage2HTN', 'Country_y',
       ...
       'gaps_percent_9', 'pop_htn_visit_10', 'gaps_hf_base_s_10',
       'gaps_percent_10', 'pop_htn_visit_11', 'gaps_hf_base_s_11',
       'gaps_percent_11', 'pop_htn_visit_12', 'gaps_hf_base_s_12',
       'gaps_percent_12'],
      dtype='object', length=114)

In [177]:
###############################################################################
##############################Physician########################################
###############################################################################

import pandas as pd
# Base scenario: one physician provides 25 consultations/day for 200 days
df['phy_s1'] = (df['phy'] * df['pop'] / 10000) * 200 * 20 * 0.1

# Define variants
pop_variants = {
    'lcl': 'pop_htn_lcl',
    'mean': 'pop_htn',
    'ucl': 'pop_htn_ucl'
}

# Storage for final results
table_dict = {
    'Income Region': []
}
for i in range(1, 13):
    table_dict[f'gap_{i}'] = []

# Get income regions
income_regions = df['Income'].unique()

# Loop through income regions
for region in income_regions:
    df_region = df[df['Income'] == region]
    row = [region]
    for i in range(1, 13):
        # Calculate total gaps under lcl, mean, ucl
        gap_lcl = (df_region[f'{pop_variants["lcl"]}'] * i - df_region['phy_s1']).sum()
        gap_mean = (df_region[f'{pop_variants["mean"]}'] * i - df_region['phy_s1']).sum()
        gap_ucl = (df_region[f'{pop_variants["ucl"]}'] * i - df_region['phy_s1']).sum()
        
    # Convert all to millions and apply negative
        gaps = sorted([-round(gap_lcl / 1_000_000), 
                    -round(gap_mean / 1_000_000), 
                    -round(gap_ucl / 1_000_000)])

        # Assign in correct order: lcl, mean, ucl
        gap_lcl_m, gap_mean_m, gap_ucl_m = gaps[0], gaps[1], gaps[2]
        
        # Format as "mean (lower–upper)"
        gap_summary = f"{round(gap_mean_m)} ({round(gap_lcl_m)} to {round(gap_ucl_m)})"
        row.append(gap_summary)
    
    # Append the row
    for idx, col in enumerate(table_dict.keys()):
        table_dict[col].append(row[idx])

# Create DataFrame
table_df = pd.DataFrame(table_dict)

# Add total row
total_row = ['Total']
for i in range(1, 13):
    gap_lcl = (df[pop_variants["lcl"]] * i - df['phy_s1']).sum() / 1_000_000
    gap_mean = (df[pop_variants["mean"]] * i - df['phy_s1']).sum() / 1_000_000
    gap_ucl = (df[pop_variants["ucl"]] * i - df['phy_s1']).sum() / 1_000_000
    # Convert all to millions and apply negative
    gaps = sorted([-round(gap_lcl), 
                    -round(gap_mean), 
                    -round(gap_ucl)])

    # Assign in correct order: lcl, mean, ucl
    gap_lcl_m, gap_mean_m, gap_ucl_m = gaps[0], gaps[1], gaps[2]
    gap_total_summary = f"{round(gap_mean_m)} ({round(gap_lcl_m)} to {round(gap_ucl_m)})"
    total_row.append(gap_total_summary)

table_df.loc[len(table_df)] = total_row

# Save to Excel
table_df.to_excel('HTN_Gaps_with_CI1.xlsx', index=False)


In [178]:
###############################################################################
##############################Non Physician########################################
###############################################################################

import pandas as pd
# Base scenario: one physician provides 25 consultations/day for 200 days
df['phy_s1'] = (df['n_phy'] * df['pop'] / 10000) * 200 * 20 * 0.1

# Define variants
pop_variants = {
    'lcl': 'pop_htn_lcl',
    'mean': 'pop_htn',
    'ucl': 'pop_htn_ucl'
}

# Storage for final results
table_dict = {
    'Income Region': []
}
for i in range(1, 13):
    table_dict[f'gap_{i}'] = []

# Get income regions
income_regions = df['Income'].unique()

# Loop through income regions
for region in income_regions:
    df_region = df[df['Income'] == region]
    row = [region]
    for i in range(1, 13):
        # Calculate total gaps under lcl, mean, ucl
        gap_lcl = (df_region[f'{pop_variants["lcl"]}'] * i - df_region['phy_s1']).sum()
        gap_mean = (df_region[f'{pop_variants["mean"]}'] * i - df_region['phy_s1']).sum()
        gap_ucl = (df_region[f'{pop_variants["ucl"]}'] * i - df_region['phy_s1']).sum()
        
        # Convert all to millions and apply negative
        gaps = sorted([-round(gap_lcl / 1_000_000), 
                    -round(gap_mean / 1_000_000), 
                    -round(gap_ucl / 1_000_000)])

        # Assign in correct order: lcl, mean, ucl
        gap_lcl_m, gap_mean_m, gap_ucl_m = gaps[0], gaps[1], gaps[2]
        
        # Format as "mean (lower–upper)"
        gap_summary = f"{round(gap_mean_m)} ({round(gap_lcl_m)} to {round(gap_ucl_m)})"
        row.append(gap_summary)
    
    # Append the row
    for idx, col in enumerate(table_dict.keys()):
        table_dict[col].append(row[idx])

# Create DataFrame
table_df = pd.DataFrame(table_dict)

# Add total row
total_row = ['Total']
for i in range(1, 13):
    gap_lcl = (df[pop_variants["lcl"]] * i - df['phy_s1']).sum() / 1_000_000
    gap_mean = (df[pop_variants["mean"]] * i - df['phy_s1']).sum() / 1_000_000
    gap_ucl = (df[pop_variants["ucl"]] * i - df['phy_s1']).sum() / 1_000_000
    # Convert all to millions and apply negative
    gaps = sorted([-round(gap_lcl), 
                    -round(gap_mean), 
                    -round(gap_ucl)])

    # Assign in correct order: lcl, mean, ucl
    gap_lcl_m, gap_mean_m, gap_ucl_m = gaps[0], gaps[1], gaps[2]
    gap_total_summary = f"{round(gap_mean_m)} ({round(gap_lcl_m)} to {round(gap_ucl_m)})"
    total_row.append(gap_total_summary)

table_df.loc[len(table_df)] = total_row

# Save to Excel
table_df.to_excel('HTN_Gaps_with_CI2.xlsx', index=False)

In [179]:
###############################################################################
##############################Team Based#######################################
###############################################################################

import pandas as pd
# Base scenario: one physician provides 25 consultations/day for 200 days
df['phy_s1'] = (df['hw'] * df['pop'] / 10000) * 200 * 20 * 0.1

# Define variants
pop_variants = {
    'lcl': 'pop_htn_lcl',
    'mean': 'pop_htn',
    'ucl': 'pop_htn_ucl'
}

# Storage for final results
table_dict = {
    'Income Region': []
}
for i in range(1, 13):
    table_dict[f'gap_{i}'] = []

# Get income regions
income_regions = df['Income'].unique()

# Loop through income regions
for region in income_regions:
    df_region = df[df['Income'] == region]
    row = [region]
    for i in range(1, 13):
        # Calculate total gaps under lcl, mean, ucl
        gap_lcl = (df_region[f'{pop_variants["lcl"]}'] * i - df_region['phy_s1']).sum()
        gap_mean = (df_region[f'{pop_variants["mean"]}'] * i - df_region['phy_s1']).sum()
        gap_ucl = (df_region[f'{pop_variants["ucl"]}'] * i - df_region['phy_s1']).sum()
        
       # Convert all to millions and apply negative
        gaps = sorted([-round(gap_lcl / 1_000_000), 
                    -round(gap_mean / 1_000_000), 
                    -round(gap_ucl / 1_000_000)])

        # Assign in correct order: lcl, mean, ucl
        gap_lcl_m, gap_mean_m, gap_ucl_m = gaps[0], gaps[1], gaps[2]
        
        # Format as "mean (lower–upper)"
        gap_summary = f"{round(gap_mean_m)} ({round(gap_lcl_m)} to {round(gap_ucl_m)})"
        row.append(gap_summary)
    
    # Append the row
    for idx, col in enumerate(table_dict.keys()):
        table_dict[col].append(row[idx])

# Create DataFrame
table_df = pd.DataFrame(table_dict)

# Add total row
total_row = ['Total']
for i in range(1, 13):
    gap_lcl = (df[pop_variants["lcl"]] * i - df['phy_s1']).sum() / 1_000_000
    gap_mean = (df[pop_variants["mean"]] * i - df['phy_s1']).sum() / 1_000_000
    gap_ucl = (df[pop_variants["ucl"]] * i - df['phy_s1']).sum() / 1_000_000
    # Convert all to millions and apply negative
    gaps = sorted([-round(gap_lcl), 
                    -round(gap_mean), 
                    -round(gap_ucl)])

    # Assign in correct order: lcl, mean, ucl
    gap_lcl_m, gap_mean_m, gap_ucl_m = gaps[0], gaps[1], gaps[2]
    gap_total_summary = f"{round(gap_mean_m)} ({round(gap_lcl_m)} to {round(gap_ucl_m)})"
    total_row.append(gap_total_summary)

table_df.loc[len(table_df)] = total_row

# Save to Excel
table_df.to_excel('HTN_Gaps_with_CI3.xlsx', index=False)